# CS6493 - Tutorial 7:  Language Model for Text Generation

In this tutorial, we will introduce how to fine tune a language model using Huggingface.

There are two types of language modeling, causal and masked. This tutorial illustrates causal language modeling.
Causal language models are frequently used for text generation. You can use these models for creative applications like
choosing your own text adventure or an intelligent coding assistant like Copilot or CodeParrot.

Causal language modeling predicts the next token in a sequence of tokens, and the model can only attend to tokens on
the left. This means the model cannot see future tokens. GPT-2 is an example of a causal language model.

This tutorial will show you how to:

1. Finetune [DistilGPT2](https://huggingface.co/distilgpt2) on a subset of the [ELI5](https://huggingface.co/datasets/eli5_category) dataset.
2. Use your finetuned model for inference.


<Tip>
You can finetune other architectures for causal language modeling following the same steps in this guide.
Choose one of the following architectures:

<!--This tip is automatically generated by `make fix-copies`, do not fill manually!-->
[OpenAI GPT](https://huggingface.co/docs/transformers/main/en/tasks/../model_doc/openai-gpt), [OpenAI GPT-2](https://huggingface.co/docs/transformers/main/en/tasks/../model_doc/gpt2), [OPT](https://huggingface.co/docs/transformers/main/en/tasks/../model_doc/opt), [Llama](https://huggingface.co/docs/transformers/main/en/tasks/../model_doc/llama), [CodeLlama](https://huggingface.co/docs/transformers/main/en/tasks/../model_doc/code_llama), etc.


<!--End of the generated tip-->

</Tip>


## Preparing Data

Again, Before you begin, make sure you have all the necessary libraries installed:


In [1]:
! pip install accelerate

In [2]:
! pip install transformers datasets evaluate

### Load ELI5 dataset

Start by loading a smaller subset of the [ sentence-transformers/eli5 dataset](https://huggingface.co/datasets/sentence-transformers/eli5) from the 🤗 Datasets library.
 This'll give you a chance to experiment and make sure everything works before spending more time training on the full dataset.

 To save the training time, we only use part of the dataset for illustration purpose. You could use the whole dataset for better model performance in your own enviornment.

Note: the original eli5_category dataset is no longer supported under the current datasets library version, as it relies on a legacy script-based loader. We use this mirror instead.


In [3]:
from datasets import load_dataset

eli5 = load_dataset("sentence-transformers/eli5", split="train[:500]")
# eli5 = load_dataset("sentence-transformers/eli5", split="train")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Split the loaded dataset into a train and test set using the [train_test_split](https://huggingface.co/docs/datasets/main/en/package_reference/main_classes#datasets.Dataset.train_test_split) method:

In [4]:
eli5 = eli5.train_test_split(test_size=0.2)

Then take a look at an example:

In [5]:
eli5["train"][0]

{'question': 'How can peristaltic movements transport liquids?',
 'answer': 'Peristalsis works like this: _URL_0_ Stick a drinking straw in a cup full of water, place your thumb over the end of the straw, and lift it out of the water so that the straw is full of water but no longer in cup. Pinch the straw flat with the thumb and forefinger of your other hand up near where your thumb is covering the end and pull down, still squeezing and covering the end with your other thumb.  That\'s basically how peristalsis works. The muscles pinch the "tube" in a small area (be it your esophagus, intestines, etc) and force its contents along with a wave motion. It pinches, pushes a little ways, and releases. The next muscle pinches it again, pushes it a little ways, and releases. This process repeats until the contents get where they\'re going.'}

Each example contains a `question` and an `answer` field. For language modeling, we only need the text content — in this case the `answer` field. Language modeling is an unsupervised task because the next word itself serves as the label.

## Preprocess

Tokenization converts raw text into a sequence of integer IDs that the model can process. Each ID corresponds to a token in the model's vocabulary — which may be a word, subword, or punctuation mark. Here we load the DistilGPT2 tokenizer and define a preprocessing function that tokenizes the `answer` field of each example:

In [6]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilgpt2")

def preprocess_function(examples):
    return tokenizer(examples["answer"])

To apply this preprocessing function over the entire dataset, use the 🤗 Datasets [map](https://huggingface.co/docs/datasets/main/en/package_reference/main_classes#datasets.Dataset.map) method. You can speed up the `map` function by setting `batched=True` to process multiple elements of the dataset at once, and increasing the number of processes with `num_proc`. Remove any columns you don't need:

In [7]:
tokenized_eli5 = eli5.map(
    preprocess_function,
    batched=True,
    num_proc=4,
    remove_columns=eli5["train"].column_names,
)

Map (num_proc=4):   0%|          | 0/400 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/100 [00:00<?, ? examples/s]

After tokenization, each example has a different sequence length — some may be very short, others very long.

To handle this efficiently, you can use the following `group_texts` function to
- concatenate all the sequences end-to-end into one long sequence, then
- split the concatenated sequence into equal-length chunks defined by `block_size`.

This ensures that every training sample has the same length and no tokens are wasted.

In [8]:
block_size = 128

def group_texts(examples):
    # Concatenate all texts.
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    # We drop the small remainder, we could add padding if the model supported it instead of this drop, you can
    # customize this part to your needs.
    if total_length >= block_size:
        total_length = (total_length // block_size) * block_size
    # Split by chunks of block_size.
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

A line-by-line explanation of `group_texts`:

1. `concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}`:

    `sum(..., [])` is a Python trick to flatten a list of lists into a single list — e.g. `sum([[1,2],[3,4]], [])` gives `[1,2,3,4]`. This is applied to every field (`input_ids`, `attention_mask`, etc.), concatenating all token sequences in the batch into one long sequence per field.

2. `total_length = len(concatenated_examples[list(examples.keys())[0]])`:

    Gets the total length of the concatenated sequence. `list(examples.keys())[0]` dynamically picks the first field name rather than hardcoding it.

3. `if total_length >= block_size: total_length = (total_length // block_size) * block_size`:

    Rounds the total length down to the nearest multiple of `block_size`, so that the remainder at the end is discarded and every resulting chunk is exactly 128 tokens long.

4. `result = {k: [t[i : i + block_size] for i in range(0, total_length, block_size)] ...}`:

    Slices each field into chunks of `block_size` tokens by stepping through the concatenated sequence with a stride of 128.

5. `result["labels"] = result["input_ids"].copy()`:

    Copies `input_ids` into a new `labels` field. The model is trained to predict the next token at each position, so the input and label are the same sequence — the one-position offset is handled automatically by `DataCollatorForLanguageModeling`.    

Apply the `group_texts` function over the entire dataset:

In [9]:
lm_dataset = tokenized_eli5.map(group_texts, batched=True, num_proc=4)

Map (num_proc=4):   0%|          | 0/400 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/100 [00:00<?, ? examples/s]

Now create a batch of examples using [DataCollatorForLanguageModeling](https://huggingface.co/docs/transformers/main/en/main_classes/data_collator#transformers.DataCollatorForLanguageModeling). It's more efficient to *dynamically pad* the
sentences to the longest length in a batch during collation, instead of padding the whole dataset to the maximum length.

Use the end-of-sequence token as the padding token and set `mlm=False`. This will use the inputs as labels shifted to the right by one element:

In [10]:
from transformers import DataCollatorForLanguageModeling

tokenizer.pad_token = tokenizer.eos_token
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

## Train

<Tip>

Transformers provides a Trainer class optimized for training 🤗 Transformers models, making it easier to start training without manually writing your own training loop. The Trainer API supports a wide range of training options and features such as logging, gradient accumulation, and mixed precision.

</Tip>

You're ready to start training your model now! Load DistilGPT2 with [AutoModelForCausalLM](https://huggingface.co/docs/transformers/main/en/model_doc/auto#transformers.AutoModelForCausalLM):

In [11]:
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer

model = AutoModelForCausalLM.from_pretrained("distilgpt2")

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


At this point, only three steps remain:

1. Define your training hyperparameters in [TrainingArguments](https://huggingface.co/docs/transformers/main/en/main_classes/trainer#transformers.TrainingArguments). The only required parameter is `output_dir` which specifies where to save your model.
2. Pass the training arguments to [Trainer](https://huggingface.co/docs/transformers/main/en/main_classes/trainer#transformers.Trainer) along with the model, datasets, and data collator.
3. Call [train()](https://huggingface.co/docs/transformers/main/en/main_classes/trainer#transformers.Trainer.train) to finetune your model.

In [12]:
training_args = TrainingArguments(
    output_dir="my_awesome_eli5_clm-model",
    eval_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lm_dataset["train"],
    eval_dataset=lm_dataset["test"],
    data_collator=data_collator,
)

`Evaluation strategy` determines when the model is evaluated on the validation dataset during training. In Hugging Face `TrainingArguments`, common options are:
  1. `no`: Never run evaluation during training
  2. `steps`: Evaluate every fixed number of training steps
  3. `epoch`: Evaluate once at the end of each epoch

`Learning rate` controls how large a step the optimizer takes when updating the model parameters during training.

`Weight decay` is a regularization technique that penalizes large model weights to help prevent overfitting.

Before fine-tuning, we first use the [evaluate()](https://huggingface.co/docs/transformers/main/en/main_classes/trainer#transformers.Trainer.evaluate) method to obtain a baseline perplexity. After training, we evaluate again to see how much the model has improved.

As a quick recap, perplexity is a common metric for language models, and measures how "surprised" the model is by a sequence of text — a lower perplexity means the model finds the text more predictable, indicating better performance. It is computed as the exponential of the average cross-entropy loss: $\text{Perplexity} = e^{\mathcal{L}}$.

In [13]:
import math

# Evaluate before fine-tuning (baseline)
baseline_results = trainer.evaluate()
print(f"Baseline Perplexity: {math.exp(baseline_results['eval_loss']):.2f}")

# Fine-tune the model
trainer.train()

eval_results = trainer.evaluate()
print(f"Fine-tuned Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Baseline Perplexity: 59.41


Epoch,Training Loss,Validation Loss,Model Preparation Time
1,No log,3.976186,0.001800
2,No log,3.960809,0.001800
3,No log,3.956490,0.001800


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuned Perplexity: 52.27


## Inference

Great, now that you've finetuned a model, you can use it for inference!

Come up with a prompt you'd like to generate text from:

In [14]:
prompt = "Somatic hypermutation allows the immune system to" # In Chinese: "体细胞超突变使免疫系统能够……"

The simplest way to try out your finetuned model for inference is to use it in a [pipeline()](https://huggingface.co/docs/transformers/main/en/main_classes/pipelines#transformers.pipeline). Instantiate a `pipeline` for text generation with your model, and pass your text to it:

In [15]:
from transformers import pipeline

generator = pipeline("text-generation", model=model, tokenizer=tokenizer)
generator(prompt)

Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': 'Somatic hypermutation allows the immune system to react to the environment and to respond to the environments quickly enough. The process of increasing the immune system\'s capacity to respond to certain conditions is known as "hypermutation."\n\n\nThe immune system is able to do its job in a way that\'s able to respond to the environment, and that\'s what it does. In order to make the system\'s response to the environment easier, the immune system must first make a better choice by ensuring that the environment is safe. So if the environment is safe, the immune system can make a good choice by ensuring that the environment is safe.\nThe next step is to have an effective immune system. After a long period of time, we can make sure that the environment is safe and that the immune system protects itself.\nHow to make sure that the environment is safe?\nAn egg is a highly porous material. It\'s called a "cellular membrane". It\'s called a "cellular membrane." It\'s ca

Note that the generated text may sound fluent but factually incorrect — this is a known limitation of language models called `hallucination`. The model has only been fine-tuned on a small subset of 500 examples for a limited number of steps, which is not enough to acquire reliable factual knowledge. A larger dataset and more training would improve both fluency and factual accuracy.

## Summary

In this tutorial, we walked through the full pipeline for causal language modeling using Hugging Face Transformers:

1. Data: Loaded the `sentence-transformers/eli5` dataset and split it into train/test sets.
2. Preprocessing: Tokenized the `answer` field, then used `group_texts` to concatenate and chunk all sequences into fixed-length blocks of 128 tokens.
3. Training: Fine-tuned a pretrained DistilGPT2 model using the `Trainer` API with a `DataCollatorForLanguageModeling` for causal language modeling.
4. Evaluation: Measured model performance using perplexity before and after fine-tuning to quantify the improvement.
5. Inference: Used a `pipeline` to generate text from a prompt with the fine-tuned model.

Keep in mind that the model was trained on a small subset of data for demonstration purposes. For better performance, consider training on the full dataset for more epochs.
